# 01 データ取得

空港の観測（METAR、Iowa Environmental Mesonet）と 1 時間ごとの気象（Open-Meteo の過去データ）を取得し、
時刻で結合して `data/processed/observations.parquet` に保存する。

- 取得・整形・結合の中身は `app/clients/` と `ml/` にある（テスト済み）。このノートブックは順に呼ぶだけ。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶんを読み、外部と通信せず、一時的な場所に書く）。
- 全期間（2018〜2025 年）を取得するときだけ、環境変数 `DATA_FETCH_MODE=full` を付けて実行する。
  取得した応答は `data/raw/` に保存し、2 回目からは外部APIを呼ばない。
- 時刻はすべて UTC。単位は摂氏・メートル。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import os
import sys
import tempfile
from functools import partial
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from app.clients.bulk import build_bulk_http_client
from app.clients.iem import IemClient
from app.clients.open_meteo_archive import OpenMeteoArchiveClient
from app.config import Settings
from ml import archive, dataset, metar, raw_store

## 設定

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    RAW_ROOT = ROOT / "data" / "raw"
    OUTPUT = ROOT / "data" / "processed" / "observations.parquet"
    YEARS = list(range(2018, 2026))
elif MODE == "sample":
    RAW_ROOT = ROOT / "tests" / "fixtures" / "data_raw"
    OUTPUT = Path(tempfile.mkdtemp()) / "observations.parquet"
    YEARS = [2025]
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

print(MODE, YEARS)

## 取得（保存済みの生データがあれば、それを読む）

In [ ]:
async def no_fetch() -> str:
    raise RuntimeError("sample mode must not call external APIs")


settings = Settings()
raw: dict[tuple[City, int], tuple[str, str]] = {}

async with (
    build_bulk_http_client(settings.iem_base_url, settings) as iem_http,
    build_bulk_http_client(settings.open_meteo_archive_base_url, settings) as archive_http,
):
    iem_client = IemClient(iem_http, settings)
    archive_client = OpenMeteoArchiveClient(archive_http, settings)
    for city in City:
        for year in YEARS:
            fetch_weather = partial(archive_client.fetch_hourly, city, year)
            fetch_metar = partial(iem_client.fetch_metar, city, year)
            if MODE == "sample":
                fetch_weather = fetch_metar = no_fetch
            weather_path = raw_store.raw_path(RAW_ROOT, "open_meteo_archive", city, year, "json")
            metar_path = raw_store.raw_path(RAW_ROOT, "iem", city, year, "csv")
            raw[(city, year)] = (
                await raw_store.load_or_fetch(weather_path, fetch_weather),
                await raw_store.load_or_fetch(metar_path, fetch_metar),
            )

len(raw)

## 整形と結合

In [ ]:
parts: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
for city in City:
    weather = pd.concat(
        [archive.parse_archive(raw[(city, year)][0]) for year in YEARS], ignore_index=True
    )
    reports = pd.concat(
        [metar.parse_metar(raw[(city, year)][1]) for year in YEARS], ignore_index=True
    )
    parts[city.value] = (weather, reports)

observations = dataset.build_dataset(parts)
dataset.validate_dataset(observations)
observations.shape

## 確認

In [ ]:
summary = observations.groupby("city").agg(
    rows=("time", "size"),
    first=("time", "min"),
    last=("time", "max"),
    without_metar=("metar", lambda column: int(column.isna().sum())),
    without_ceiling=("ceiling_m", lambda column: int(column.isna().sum())),
)
summary

## 保存

In [ ]:
dataset.save_dataset(observations, OUTPUT)
reloaded = dataset.load_dataset(OUTPUT)
pd.testing.assert_frame_equal(reloaded, observations)
print(OUTPUT.name, len(reloaded))